# 01 — Generate AUDIO-grounded Tool-Calling Data

**Run on a GPU Modal notebook** (Mimi encoding needs GPU; TTS is CPU/network).

The key fix over text-only training: the user's question is placed in the
**audio** rows of the code tensor, so Moshi learns to emit `<|tool_call|>` when
it *hears* a request — which is what actually happens at inference.

Per example we build `codes[17, T]`:

| rows | stream | content |
|------|--------|---------|
| `0` | text monologue | PAD while listening, then `<|tool_call|>…` + spoken reply |
| `1:9` | Moshi audio | silence (Mimi-encoded) |
| `9:17` | **user audio** | **edge-tts speech of the question, Mimi-encoded** |

Pipeline: `gen_tool_data.examples()` → edge-tts (many voices) → Mimi encode →
`codes[17,T]` → cache as one `.pt` on HuggingFace. Notebook 02 loads it and trains.

In [ ]:
# Install into the EXACT python running this kernel
import sys
!{sys.executable} -m pip install edge-tts sphn nest_asyncio sentencepiece huggingface_hub einops numpy tqdm datasets soundfile -q
import importlib
for m in ["edge_tts", "sphn", "nest_asyncio", "sentencepiece", "einops",
          "huggingface_hub", "tqdm", "datasets", "soundfile"]:
    importlib.import_module(m)
print("All deps importable ✓")

In [ ]:
import gc, os, shutil, subprocess, asyncio, tempfile
from pathlib import Path
import numpy as np
import torch

# ── Clone repo ────────────────────────────────────────────────────────────────
REPO = Path("/repo")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone",
    "https://github.com/syedfahimabrar/moshi-D-gu.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "moshi"))
sys.path.insert(0, str(REPO / "notebooks"))

import sentencepiece as spm
from moshi.models import loaders
import gen_tool_data as G

# ── Config ────────────────────────────────────────────────────────────────────
HF_PATCHED_REPO = "abrarfahim/moshi-tool-patched"   # tokenizer source
HF_DATA_REPO    = "abrarfahim/moshi-tool-audio"     # where we cache the dataset
HF_TOKEN        = os.environ.get("HF_TOKEN", None)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SR     = 24000          # Mimi sample rate
FRAME  = 1920          # samples per 12.5 Hz frame (24000 / 12.5)

# edge-tts voices — variety of accents/genders so triggering generalises
VOICES = [
    "en-US-AriaNeural", "en-US-GuyNeural", "en-US-JennyNeural",
    "en-US-MichelleNeural", "en-US-ChristopherNeural", "en-US-EricNeural",
    "en-GB-SoniaNeural", "en-GB-RyanNeural", "en-GB-LibbyNeural",
    "en-AU-NatashaNeural", "en-AU-WilliamNeural",
    "en-IN-NeerjaNeural", "en-IN-PrabhatNeural",
    "en-CA-ClaraNeural", "en-CA-LiamNeural",
]
print(f"Device: {DEVICE} | voices: {len(VOICES)}")

In [ ]:
from huggingface_hub import hf_hub_download

# Tokenizer (patched, 32004 tokens)
tok_path = hf_hub_download(HF_PATCHED_REPO, "tokenizer_spm_32k_3.model", token=HF_TOKEN)
tok = spm.SentencePieceProcessor(); tok.Load(tok_path)
assert tok.get_piece_size() == 32004, tok.get_piece_size()
assert tok.id_to_piece(G.TOOL_CALL_ID) == "<|tool_call|>"
print(f"Tokenizer ✓ ({tok.get_piece_size()} tokens)")

# Mimi audio codec (from PersonaPlex), 8 codebooks
mimi_path = hf_hub_download(loaders.DEFAULT_REPO, loaders.MIMI_NAME, token=HF_TOKEN)
mimi = loaders.get_mimi(mimi_path, device=DEVICE)
mimi.eval()

# Verify N*FRAME samples -> N frames, and 8 codebooks
with torch.no_grad():
    probe = mimi.encode(torch.zeros(1, 1, FRAME * 10, device=DEVICE))
print(f"Mimi ✓  encode(10 frames) -> {tuple(probe.shape)}  (expect [1, 8, 10])")
assert probe.shape[1] == 8

In [ ]:
import edge_tts, sphn, nest_asyncio
nest_asyncio.apply()   # allow asyncio.run inside the Jupyter event loop

def _resample(wav, src, dst):
    if src == dst:
        return wav.astype("float32")
    n = int(round(len(wav) * dst / src))
    x  = np.linspace(0, 1, len(wav), endpoint=False)
    xn = np.linspace(0, 1, n, endpoint=False)
    return np.interp(xn, x, wav).astype("float32")

_tts_cache = {}   # (text, voice) -> waveform; queries repeat, so this saves calls

async def _save_tts(text, voice, path):
    await edge_tts.Communicate(text, voice).save(path)

def tts_wav(text, voice):
    key = (text, voice)
    if key in _tts_cache:
        return _tts_cache[key]
    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as fh:
        path = fh.name
    asyncio.run(_save_tts(text, voice, path))
    wav, sr = sphn.read(path)        # wav: [C, N] float32
    os.unlink(path)
    if wav.ndim > 1:
        wav = wav.mean(axis=0)
    wav = _resample(wav, sr, SR)
    _tts_cache[key] = wav
    return wav

# quick smoke test
_w = tts_wav("what time is it", VOICES[0])
print(f"TTS ✓  'what time is it' -> {len(_w)} samples ({len(_w)/SR:.2f}s)")

In [ ]:
def _pad_to_frames(wav):
    """Pad a waveform up to a whole number of frames; return (wav, n_frames)."""
    n = len(wav)
    fr = int(np.ceil(n / FRAME)) if n > 0 else 0
    out = np.zeros(fr * FRAME, dtype="float32")
    out[:n] = wav
    return out, fr

def _silence(frames):
    return np.zeros(frames * FRAME, dtype="float32")

@torch.no_grad()
def _encode(wav):
    t = torch.from_numpy(wav).to(DEVICE)[None, None]   # [1,1,N]
    return mimi.encode(t)[0].cpu()                      # [8, T]

def _fit(codes, T):
    """Trim or pad (repeat last/silence frame) audio codes to exactly T frames."""
    Ta = codes.shape[1]
    if Ta < T:
        codes = torch.cat([codes, codes[:, -1:].repeat(1, T - Ta)], dim=1)
    elif Ta > T:
        codes = codes[:, :T]
    return codes

def render(ex, voice):
    """Abstract Example -> dataset record (codes + mask + inspectable fields)."""
    user_parts, text, mask = [], [], []
    queries, replies = [], []
    for turn in ex.turns:
        if turn.query is None:                         # pure-silence turn
            N = int(np.random.randint(20, 50))
            k = int(np.random.randint(N // 3, N // 2))
            text += [G.PAD_ID] * N
            mask += [0] * k + [1] * (N - k)            # train tail to stay PAD
            user_parts.append(_silence(N))
            continue
        qwav, fq = _pad_to_frames(tts_wav(turn.query, voice))
        gap = int(np.random.randint(2, 6))
        emit_t, emit_m = G.render_emit(turn, tok)
        E = len(emit_t)
        text += [G.PAD_ID] * (fq + gap) + emit_t       # listen, pause, then emit
        mask += [0] * (fq + gap) + emit_m
        user_parts.append(np.concatenate([qwav, _silence(gap + E)]))
        queries.append(turn.query)
        replies.append(turn.reply)

    user_wav = np.concatenate(user_parts) if user_parts else _silence(1)
    T = len(text)
    user_codes  = _fit(_encode(user_wav), T)
    moshi_codes = _fit(_encode(_silence(T)), T)        # Moshi audio = silence

    codes = torch.zeros(17, T, dtype=torch.long)
    codes[0]    = torch.tensor(text, dtype=torch.long)
    codes[1:9]  = moshi_codes
    codes[9:17] = user_codes
    return {
        "type":  ex.type,
        "query": " | ".join(queries),
        "reply": " | ".join(replies),
        "audio": {"array": user_wav.astype("float32"), "sampling_rate": SR},
        "codes": codes.to(torch.int32).tolist(),       # 17 x T (Arrow-friendly)
        "mask":  [int(m) for m in mask],
        "voice": voice,
    }

# smoke test one example
_ex = next(e for e in G.examples() if e.type == "time")
_r = render(_ex, VOICES[0])
print(f"render ✓  type={_r['type']}  query={_r['query']!r}")
print(f"  codes={len(_r['codes'])}x{len(_r['codes'][0])}  mask={len(_r['mask'])}  "
      f"audio={len(_r['audio']['array'])} samples")

In [ ]:
from tqdm.auto import tqdm

exs = G.examples(seed=42)
np.random.seed(42)
records = []
for i, ex in enumerate(tqdm(exs, desc="TTS + Mimi encode")):
    voice = VOICES[i % len(VOICES)]     # rotate voices for even coverage
    records.append(render(ex, voice))

from collections import Counter
print(f"\nBuilt {len(records)} records")
print("Breakdown:", dict(Counter(r["type"] for r in records)))
print(f"Avg frames: {sum(len(r['mask']) for r in records)/len(records):.0f}")
print(f"Max frames: {max(len(r['mask']) for r in records)}")
print(f"Unique TTS clips cached: {len(_tts_cache)}")

In [ ]:
# Build a proper HF dataset (browsable, playable audio) and push to the Hub
from datasets import Dataset, Features, Value, Sequence, Audio

features = Features({
    "type":  Value("string"),
    "query": Value("string"),
    "reply": Value("string"),
    "voice": Value("string"),
    "audio": Audio(sampling_rate=SR),                       # playable in the viewer
    "codes": Sequence(Sequence(Value("int32"))),           # 17 x T training cache
    "mask":  Sequence(Value("int8")),
})

ds = Dataset.from_list(records, features=features)
print(ds)

ds.push_to_hub(HF_DATA_REPO, private=True, token=HF_TOKEN)
print(f"\nPushed → https://huggingface.co/datasets/{HF_DATA_REPO}")
print("Next: run notebook 02 (it loads this dataset and trains).")